# Feature Engineering for PM2.5 Prediction

In the previous notebook, we cleaned the Delhi air pollution dataset and performed exploratory data analysis.

In this notebook, we focus on **feature engineering**, which involves creating new variables that can help machine learning models better understand patterns in the data.

Key steps in this notebook:

• Load cleaned dataset  
• Create time-based features  
• Create lag features  
• Create rolling statistics  
• Prepare dataset for machine learning models

In [1]:
# Importing necessary libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading the cleaned dataset from Notebook 1

df = pd.read_csv("../data/processed/delhi_cleaned.csv")

# Convert date column again (CSV reload resets dtype)

df["date"] = pd.to_datetime(df["date"])

df.head()

,date,co,no,no2,o3,so2,pm2_5,pm10,nh3,pm2_5_7day_avg,pm2_5_30day_avg,year,month
0,2020-11-25 01:00:00,2616.88,2.18,70.60,13.59,38.62,364.61,411.73,28.63,NaN,NaN,2020,11
1,2020-11-25 02:00:00,3631.59,23.25,89.11,0.33,54.36,420.96,486.21,41.04,NaN,NaN,2020,11
2,2020-11-25 03:00:00,4539.49,52.75,100.08,1.11,68.67,463.68,541.95,49.14,NaN,NaN,2020,11
3,2020-11-25 04:00:00,4539.49,50.96,111.04,6.44,78.20,454.81,534.00,48.13,NaN,NaN,2020,11
4,2020-11-25 05:00:00,4379.27,42.92,117.90,17.17,87.74,448.14,529.19,46.61,NaN,NaN,2020,11


In [3]:
# Ensuring the dataset is ordered chronologically

df = df.sort_values("date")

In [4]:
# Extracting useful time-based features

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek

### Lag Features

Lag features represent historical pollution values.

Air pollution usually follows temporal patterns,
which means today's pollution level depends on
pollution levels from previous days.

These lag features help the model capture that pattern.

These features allow the model to learn seasonal pollution patterns.

For example:
- Winter months often show higher pollution levels.
- Weekdays vs weekends may show different traffic pollution patterns.

In [5]:
# Creating lag features for PM2.5

df["pm2_5_lag1"] = df["pm2_5"].shift(1)
df["pm2_5_lag2"] = df["pm2_5"].shift(2)
df["pm2_5_lag3"] = df["pm2_5"].shift(3)

Lag features represent pollution levels from previous days.

Example:
- pm2_5_lag1 → pollution yesterday
- pm2_5_lag2 → pollution 2 days ago

These features are extremely important for **time-series prediction models**.

In [6]:
# Rolling mean features

df["pm2_5_roll7"] = df["pm2_5"].rolling(window=7).mean()
df["pm2_5_roll14"] = df["pm2_5"].rolling(window=14).mean()

In [7]:
# Rolling standard deviation

df["pm2_5_std7"] = df["pm2_5"].rolling(window=7).std()

Rolling statistics summarize recent pollution behavior.

Examples:
- 7-day rolling mean shows weekly trend
- Rolling std shows how volatile pollution levels are

In [8]:
# Removing rows with missing values

df = df.dropna()

df.head()

,date,co,no,no2,o3,so2,pm2_5,pm10,nh3,pm2_5_7day_avg,...,year,month,day,day_of_week,pm2_5_lag1,pm2_5_lag2,pm2_5_lag3,pm2_5_roll7,pm2_5_roll14,pm2_5_std7
29,2020-11-26 06:00:00,2109.53,23.69,67.86,33.26,108.72,208.00,253.85,26.60,278.282857,...,2020,11,26,3,264.39,300.52,337.80,278.282857,297.438571,41.948721
30,2020-11-26 07:00:00,1094.82,9.95,49.35,66.52,90.60,115.32,139.51,12.79,257.607143,...,2020,11,26,3,208.00,264.39,300.52,257.607143,277.496429,75.044687
31,2020-11-26 08:00:00,894.55,7.49,42.50,82.25,83.92,92.77,111.59,10.13,232.562857,...,2020,11,26,3,115.32,208.00,264.39,232.562857,257.765714,97.006288
32,2020-11-26 09:00:00,881.20,7.26,44.55,85.12,92.51,88.53,107.12,10.64,201.047143,...,2020,11,26,3,92.77,115.32,208.00,201.047143,240.697143,103.593793
33,2020-11-26 10:00:00,1308.44,9.16,63.06,67.23,100.14,106.99,132.10,17.23,168.074286,...,2020,11,26,3,88.53,92.77,115.32,168.074286,226.638571,88.435457


In [9]:
# Selecting input features

features = [
    "month",
    "day",
    "day_of_week",
    "pm2_5_lag1",
    "pm2_5_lag2",
    "pm2_5_lag3",
    "pm2_5_roll7",
    "pm2_5_roll14",
    "pm2_5_std7"
]

X = df[features]
y = df["pm2_5"]

In [10]:
# Saving final dataset for modeling

feature_df = df[features + ["pm2_5"]]

feature_df.to_csv("../data/processed/delhi_features.csv", index=False)

In this notebook we created several new features to improve model performance.

These include:

• Time-based features  
• Lag features  
• Rolling statistics  

The final dataset has been saved and will be used in the next notebook to train machine learning models.

### Feature Engineering Rationale

Temporal features such as month and day were created because
air pollution often follows seasonal patterns.

Lag features were also included to capture the effect of
previous pollution levels on current PM2.5 values.

These engineered features help machine learning models
better capture temporal trends and environmental patterns.